# Day 98: Capstone: Deploy to Home Lab Cluster

**Layer:** Capstone


## Concept Overview

Deploy the inference server to spark-01 and spark-02 with Nginx load balancing, health-check-gated traffic switching, and Prometheus + Grafana monitoring. The full production stack running on real hardware.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Deployment Manifest

Generate the deployment scripts and Nginx config for the home lab cluster.


In [ ]:
print("Home lab deployment scripts:")
print()
print("=== docker-compose.yml ===")
print("""
version: '3.9'
services:
  inference-server:
    image: inference-server:latest
    runtime: nvidia
    environment:
      - NVIDIA_VISIBLE_DEVICES=all
      - MODEL_PATH=/models/llama-3-8b-int8
    ports:
      - "8000:8000"
    volumes:
      - /data/models:/models:ro
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 10s
      retries: 6
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: all
              capabilities: [gpu]

  prometheus:
    image: prom/prometheus
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
    ports:
      - "9090:9090"

  grafana:
    image: grafana/grafana
    ports:
      - "3000:3000"
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=admin
""")
print("=== nginx.conf (load balancer) ===")
print("""
upstream inference {
    least_conn;
    server spark-01:8000 max_fails=3 fail_timeout=30s;
    server spark-02:8000 max_fails=3 fail_timeout=30s;
}
server {
    listen 80;
    location / { proxy_pass http://inference; proxy_read_timeout 300; }
    location /health { proxy_pass http://inference; }
}
""")
print("Deploy: docker compose up -d --scale inference-server=1 on each spark node.")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Deploy the inference server to spark-01 and spark-02 with Nginx load balancing, health-check-gated traffic switching, and Prometheus + Grafana monitoring.
- Deploy the inference server to spark-01 and spark-02 with Nginx load balancing, ....
- Day 98 implementation complete.
